In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# Facility Constants
PUE = 1.2
IDLE_FRACTION = 0.20 # Servers draw 20% max power even when idle
SCENARIOS = ['Normal Day', 'High Usage (Spike)', 'Weekend/Batch']
TIME_STEPS = 96 # 15-minute intervals over 24 hours
SIM_ITERATIONS = 1000

# Time array (0 to 24 hours)
time_hours = np.linspace(0, 24, TIME_STEPS)

def generate_base_utilization(scenario):
    """Generate the mean utilization curve U(t) for a given scenario."""
    if scenario == 'Normal Day':
        u = 0.4 + 0.3 * np.sin(np.pi * (time_hours - 6) / 12) + 0.1 * np.sin(np.pi * (time_hours - 14) / 6)
    elif scenario == 'High Usage (Spike)':
        u = 0.7 + 0.25 * np.sin(np.pi * (time_hours - 8) / 14)
    else: # Weekend/Batch
        u = 0.3 + 0.1 * np.sin(np.pi * (time_hours - 12) / 12)
    return np.clip(u, 0, 1)

def run_monte_carlo_normalized(scenario, iterations=SIM_ITERATIONS):
    # Using 100.0 as the MAX_POWER to represent percentage (0% to 100%)
    max_power_pct = 100.0
    max_it_power = max_power_pct / PUE
    base_it_power = max_it_power * IDLE_FRACTION

    mean_u = generate_base_utilization(scenario)
    volatility = 0.15

    simulations = np.zeros((iterations, TIME_STEPS))

    for i in range(iterations):
        noise = np.random.normal(0, volatility, TIME_STEPS)
        smoothed_noise = pd.Series(noise).rolling(window=4, min_periods=1).mean().values

        u_sim = mean_u + smoothed_noise
        u_sim = np.clip(u_sim, 0.0, 1.0)

        it_power = base_it_power + (max_it_power - base_it_power) * u_sim
        total_power = it_power * PUE

        # Hard clip at 100% capacity
        total_power = np.clip(total_power, 0, 100.0)
        simulations[i, :] = total_power

    p5 = np.percentile(simulations, 5, axis=0)
    p50 = np.percentile(simulations, 50, axis=0)
    p95 = np.percentile(simulations, 95, axis=0)

    return p5, p50, p95

# Plotting Execution
fig, ax = plt.subplots(figsize=(12, 6))

colors = {'Normal Day': 'blue', 'High Usage (Spike)': 'red', 'Weekend/Batch': 'green'}

for scenario in SCENARIOS:
    p5, p50, p95 = run_monte_carlo_normalized(scenario)

    # Plot Median
    ax.plot(time_hours, p50, label=f'{scenario} (Median)', color=colors[scenario], linewidth=2)
    # Plot 90% Confidence Interval
    ax.fill_between(time_hours, p5, p95, color=colors[scenario], alpha=0.15)

ax.set_title('Stochastic 24-Hour Power Demand \n(Normalized Across All Scaling Phases)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day (CST)', fontsize=12)
ax.set_ylabel('Power Demand (% of Total Facility Capacity)', fontsize=12)
ax.set_xlim(0, 24)
ax.set_ylim(0, 110)
ax.axhline(100, color='black', linestyle='--', linewidth=2, label='100% Facility Cap (25MW, 100MW, 250MW)')

# Formatting
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, linestyle=':', alpha=0.7)
plt.xticks(np.arange(0, 25, 2))
plt.yticks(np.arange(0, 111, 10))

plt.savefig('demand_monte_carlo_normalized.png', dpi=300, bbox_inches='tight')
plt.show()

TABLE 1: LCOE Input Parameters (2026 Basis)
-------------------------------------------------------------------------------------------------------------------
           Technology  CapEx ($/kW)  Fixed O&M ($/kW-yr)  Var O&M ($/MWh)  Fuel ($/MWh)  Cap Factor  Lifespan (Yrs)
  ERCOT Grid (345 kV)           150                    5                0         75.00       1.000              20
    Utility Solar PPA             0                    0                0         35.00       0.280              20
   RICE (Natural Gas)          1200                    0               15         29.75       0.900              20
  Enhanced Geothermal          4500                  115                0          0.00       0.950              30
Small Modular Reactor          7500                  120                0          8.32       0.950              40
        Microturbines          1500                    0               20         40.25       0.900              20
   Hybrid BESS (LCOS)       